# IDS on CICIDS2017 — experiments (free Colab)

Chạy hết trên Google Colab miễn phí. Kết quả (2 bảng) dán thẳng vào `main.tex`.

**Quy trình:** tải dữ liệu → làm sạch → chọn đặc trưng → train 4 model → xuất bảng metrics + bảng cost.

> Nguyên tắc: chỉ điền số THẬT do notebook này sinh ra vào paper. Không bịa.

## 0. Cài thư viện

In [ ]:
!pip -q install xgboost imbalanced-learn scikit-learn pandas numpy matplotlib


## 1. Lấy dữ liệu CICIDS2017

Bộ `MachineLearningCVE` gồm 8 file CSV (nguồn: UNB CIC, hoặc mirror trên Kaggle).
Chọn 1 trong 2 cách rồi đặt `DATA_DIR` trỏ tới thư mục chứa các file `*.csv`.

- **Kaggle:** upload `kaggle.json` rồi `kaggle datasets download -d <mirror>` (vd. `cicdataset/cicids2017`).
- **Thủ công:** tải CSV về Google Drive, mount Drive, trỏ `DATA_DIR` vào đó.

In [ ]:
import glob, os

# TODO: đổi đường dẫn cho đúng nơi bạn để 8 file CSV của CICIDS2017
DATA_DIR = '/content/cicids2017'

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print('Found', len(csv_files), 'CSV files')
for f in csv_files:
    print(' -', os.path.basename(f))
assert csv_files, 'Chưa thấy file CSV — kiểm tra DATA_DIR'


## 2. Nạp & làm sạch

In [ ]:
import pandas as pd, numpy as np

df = pd.concat((pd.read_csv(f, low_memory=False) for f in csv_files), ignore_index=True)
df.columns = df.columns.str.strip()              # bỏ khoảng trắng tên cột

label_col = 'Label' if 'Label' in df.columns else df.columns[-1]
df = df.replace([np.inf, -np.inf], np.nan).dropna()
df = df.drop_duplicates()

print('Shape:', df.shape)
print(df[label_col].value_counts())


## 3. Encode nhãn (nhị phân BENIGN / ATTACK) + tách X, y

In [ ]:
y = (df[label_col].astype(str).str.upper() != 'BENIGN').astype(int)   # 1 = attack
X = df.drop(columns=[label_col]).select_dtypes(include=[np.number]).copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
print('X:', X.shape, '| attack ratio: %.4f' % y.mean())


## 4. Split + chuẩn hoá + chọn đặc trưng (top-k theo RF importance)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

SEED = 42
K = 20  # TODO: số đặc trưng giữ lại — ghi vào paper

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr); X_te_s = scaler.transform(X_te)

rf_sel = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED).fit(X_tr_s, y_tr)
top_idx = np.argsort(rf_sel.feature_importances_)[::-1][:K]
top_feats = X.columns[top_idx].tolist()
print('Top-%d features:' % K, top_feats)

X_tr_k = X_tr_s[:, top_idx]; X_te_k = X_te_s[:, top_idx]


## 5. (Tuỳ chọn) cân bằng lớp bằng SMOTE trên tập train

In [ ]:
from imblearn.over_sampling import SMOTE
USE_SMOTE = True   # TODO: ghi rõ trong paper có dùng SMOTE hay không
if USE_SMOTE:
    X_tr_k, y_tr = SMOTE(random_state=SEED).fit_resample(X_tr_k, y_tr)
    print('After SMOTE:', X_tr_k.shape, '| attack ratio: %.4f' % np.mean(y_tr))


## 6. Train + đánh giá 4 model (đo cả thời gian)

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, n_jobs=-1),
    'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED),
    'XGBoost':             XGBClassifier(n_estimators=300, max_depth=8, tree_method='hist',
                                         eval_metric='logloss', random_state=SEED),
    'Deep model (MLP)':    MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=50, random_state=SEED),
}

perf_rows, cost_rows = [], []
for name, clf in models.items():
    t0 = time.time(); clf.fit(X_tr_k, y_tr); train_t = time.time() - t0
    t0 = time.time(); y_pred = clf.predict(X_te_k); infer_ms = (time.time() - t0) / len(X_te_k) * 1000
    try:
        y_score = clf.predict_proba(X_te_k)[:, 1]; auc = roc_auc_score(y_te, y_score)
    except Exception:
        auc = float('nan')
    perf_rows.append({'Model': name,
                      'Accuracy':  round(accuracy_score(y_te, y_pred), 4),
                      'Precision': round(precision_score(y_te, y_pred), 4),
                      'Recall':    round(recall_score(y_te, y_pred), 4),
                      'macro-F1':  round(f1_score(y_te, y_pred, average='macro'), 4),
                      'ROC-AUC':   round(auc, 4)})
    cost_rows.append({'Model': name,
                      'Train time (s)':    round(train_t, 2),
                      'Inference (ms/sample)': round(infer_ms, 4)})
    print('done:', name)

perf = pd.DataFrame(perf_rows); cost = pd.DataFrame(cost_rows)
display(perf); display(cost)


## 7. Xuất ra LaTeX (dán thẳng vào main.tex, thay 2 bảng `--`)

In [ ]:
print('% ==== Table: detection performance (\\label{tab:results}) ====')
print(perf.to_latex(index=False, escape=False))
print('% ==== Table: computational cost (\\label{tab:cost}) ====')
print(cost.to_latex(index=False, escape=False))
perf.to_csv('results_performance.csv', index=False)
cost.to_csv('results_cost.csv', index=False)
print('Saved results_performance.csv, results_cost.csv')


## 8. (Tuỳ chọn) Confusion matrix + ROC cho model tốt nhất
Lưu hình vào `figures/` để đưa vào paper.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

best_name = perf.sort_values('macro-F1', ascending=False).iloc[0]['Model']
best = models[best_name]
print('Best by macro-F1:', best_name)

ConfusionMatrixDisplay.from_estimator(best, X_te_k, y_te); plt.title(best_name); plt.tight_layout()
plt.savefig('confusion_matrix.pdf'); plt.show()
try:
    RocCurveDisplay.from_estimator(best, X_te_k, y_te); plt.title('ROC — ' + best_name)
    plt.tight_layout(); plt.savefig('roc_curve.pdf'); plt.show()
except Exception as e:
    print('ROC skip:', e)
